# 06: Out-of-sample Sharpe with benchmarks

Companion to Section 7 of the paper. Quantifies the in-sample/out-of-sample
Sharpe gap for the brute-force oracle and SLH, with two benchmarks:
* **Top-$k$-$\hat\mu$**: pick top $k$ stocks by $\hat\mu_i$ alone (no covariance adjustment).
* **Eq-$n$**: equal-weight on all $n$ stocks in the cell (no selection).

Conventions exactly match `02_oracle_baselines.ipynb` and `05_hybrid_algorithm.ipynb`:
$\gamma = 10$, 80/20 train/test split, $\sqrt{252}$ Sharpe annualization, 5 random asset draws per $(n,k)$, same seed pattern.

In [ ]:
from __future__ import annotations

import time
from itertools import combinations
from math import comb
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
FIG_DIR = DATA_DIR / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

GAMMA = 10.0
TRAIN_FRAC = 0.80
TRADING_DAYS = 252
N_INSTANCES = 5
GRID = [(20, 5), (30, 5), (30, 8), (40, 5), (50, 5)]

R_full = np.load(DATA_DIR / "returns.npy")
T_full, n_full = R_full.shape
T_tr = int(TRAIN_FRAC * T_full)
T_te = T_full - T_tr
R_train_full = R_full[:T_tr]
R_test_full = R_full[T_tr:]
print(f"Full panel:  T={T_full}, n={n_full}")
print(f"Train:       T_tr={T_tr}")
print(f"Test:        T_te={T_te}")

## Core utilities (mirroring Notebook 02)

In [ ]:
def estimate_mu_sigma(R_train):
    mu = R_train.mean(axis=0)
    Sigma = np.cov(R_train, rowvar=False)
    return mu, Sigma


def J_from_support(S, mu, Sigma, gamma=GAMMA):
    k = len(S)
    sub_mu = mu[S].sum()
    sub_var = Sigma[np.ix_(S, S)].sum()
    return sub_mu / k - gamma * sub_var / (2.0 * k * k)


def annualized_sharpe(R_period, S):
    """Equal-weight Sharpe on subset S, annualized by sqrt(252)."""
    pf = R_period[:, S].mean(axis=1)
    if pf.std(ddof=1) < 1e-12:
        return 0.0
    return float(np.sqrt(TRADING_DAYS) * pf.mean() / pf.std(ddof=1))


def annualized_sharpe_series(pf):
    """Sharpe for a precomputed daily-return series (for Eq-n)."""
    if pf.std(ddof=1) < 1e-12:
        return 0.0
    return float(np.sqrt(TRADING_DAYS) * pf.mean() / pf.std(ddof=1))


def sample_instance(n, seed):
    rng = np.random.default_rng(seed)
    idx = rng.choice(n_full, size=n, replace=False)
    Rtr = R_train_full[:, idx]
    Rte = R_test_full[:, idx]
    mu, Sigma = estimate_mu_sigma(Rtr)
    return mu, Sigma, Rtr, Rte


def sharpe_se(sh, T):
    """Lo (2002) iid-Gaussian Sharpe standard error: sqrt((1 + 0.5*Sh^2) / T)."""
    return float(np.sqrt((1.0 + 0.5 * sh * sh) / T))

## Brute-force oracle (verbatim from Notebook 02)

In [ ]:
BRUTE_FORCE_CAP = 50_000_000


def _chunked_combinations(n, k, chunk):
    it = combinations(range(n), k)
    while True:
        block = []
        for _ in range(chunk):
            try:
                block.append(next(it))
            except StopIteration:
                if block:
                    yield np.asarray(block, dtype=np.int32)
                return
        yield np.asarray(block, dtype=np.int32)


def brute_force_oracle(mu, Sigma, k, gamma=GAMMA, chunk=200_000, cap=BRUTE_FORCE_CAP):
    n = len(mu)
    total = comb(n, k)
    if total > cap:
        raise ValueError(f"C({n},{k}) = {total:,} exceeds cap")
    best_J = -np.inf
    best_S = None
    for block in _chunked_combinations(n, k, chunk):
        mu_S = mu[block].sum(axis=1)
        sub = Sigma[block[:, :, None], block[:, None, :]]
        var_S = sub.sum(axis=(1, 2))
        Js = mu_S / k - gamma * var_S / (2.0 * k * k)
        local_max = int(np.argmax(Js))
        if Js[local_max] > best_J:
            best_J = float(Js[local_max])
            best_S = block[local_max].copy()
    return np.sort(best_S), float(best_J)

## SLH (verbatim from Notebook 05)

In [ ]:
def greedy_swap_polish(S0, mu, Sigma, gamma=GAMMA, max_iter=500):
    n = len(mu); k = len(S0)
    S = np.sort(np.asarray(S0, dtype=np.int64)).copy()
    in_S = np.zeros(n, dtype=bool); in_S[S] = True
    s_row = Sigma[:, S].sum(axis=1).astype(float)
    diag = np.diag(Sigma).astype(float)
    inv_k = 1.0 / k
    inv_2k2 = 1.0 / (2.0 * k * k)
    for it in range(max_iter):
        out_idx = np.where(~in_S)[0]
        if len(out_idx) == 0:
            break
        delta_mu = mu[out_idx][None, :] - mu[S][:, None]
        delta_var = (
            -2.0 * s_row[S][:, None] + diag[S][:, None]
            + 2.0 * s_row[out_idx][None, :] - 2.0 * Sigma[np.ix_(S, out_idx)]
            + diag[out_idx][None, :]
        )
        delta_J = delta_mu * inv_k - gamma * delta_var * inv_2k2
        flat = int(np.argmax(delta_J))
        bi, bj = divmod(flat, len(out_idx))
        if float(delta_J[bi, bj]) <= 1e-15:
            break
        i_remove = int(S[bi]); j_add = int(out_idx[bj])
        S[bi] = j_add
        S = np.sort(S)
        in_S[i_remove] = False; in_S[j_add] = True
        s_row += Sigma[:, j_add] - Sigma[:, i_remove]
    return S


def slh_support(mu, Sigma, k, gamma=GAMMA, eta=None):
    n = len(mu)
    if eta is None:
        eta = 1e-3 * np.trace(Sigma) / n
    A = gamma * Sigma + eta * np.eye(n)
    w_init = np.linalg.solve(A, mu)
    S0 = np.sort(np.argsort(-np.abs(w_init))[:k])
    return greedy_swap_polish(S0, mu, Sigma, gamma)

## Benchmarks: Top-$k$-$\hat\mu$ and Eq-$n$

In [ ]:
def top_k_mu_support(mu, k):
    """Top-k stocks by hat-mu_i alone (no covariance adjustment)."""
    return np.sort(np.argsort(-mu)[:k])


def eq_n_returns(R_period):
    """Equal-weighted across all n stocks in the panel."""
    return R_period.mean(axis=1)

## Main experiment loop

In [ ]:
rows = []
for (n, k) in GRID:
    print(f"\n=== (n, k) = ({n}, {k}),  C(n,k) = {comb(n,k):,} ===")
    for inst in range(N_INSTANCES):
        seed = 100 * n + 7 * k + inst  # same seeding pattern as 02_oracle_baselines
        mu, Sigma, Rtr, Rte = sample_instance(n, seed=seed)

        # 1. Brute-force oracle
        t0 = time.perf_counter()
        S_star, J_star = brute_force_oracle(mu, Sigma, k)
        oracle_dt = time.perf_counter() - t0

        # 2. SLH
        S_slh = slh_support(mu, Sigma, k)

        # 3. Top-k-mu (no covariance)
        S_topmu = top_k_mu_support(mu, k)

        # 4. Eq-n (no selection)
        eq_tr_ret = eq_n_returns(Rtr)
        eq_te_ret = eq_n_returns(Rte)

        for alg_name, S in [("oracle", S_star), ("slh", S_slh), ("topmu", S_topmu)]:
            sh_is = annualized_sharpe(Rtr, S)
            sh_oos = annualized_sharpe(Rte, S)
            rows.append({
                "n": n, "k": k, "inst": inst, "alg": alg_name,
                "sharpe_is": sh_is,
                "sharpe_oos": sh_oos,
                "sharpe_oos_se": sharpe_se(sh_oos, T_te),
                "delta_rel": (sh_is - sh_oos) / sh_is if abs(sh_is) > 1e-9 else np.nan,
                "J": J_from_support(S, mu, Sigma),
                "J_gap_bp": (J_star - J_from_support(S, mu, Sigma)) * 1e4,
                "match_oracle": int(np.array_equal(np.sort(S), np.sort(S_star))),
            })

        sh_is_eq = annualized_sharpe_series(eq_tr_ret)
        sh_oos_eq = annualized_sharpe_series(eq_te_ret)
        rows.append({
            "n": n, "k": k, "inst": inst, "alg": "eqn",
            "sharpe_is": sh_is_eq,
            "sharpe_oos": sh_oos_eq,
            "sharpe_oos_se": sharpe_se(sh_oos_eq, T_te),
            "delta_rel": (sh_is_eq - sh_oos_eq) / sh_is_eq if abs(sh_is_eq) > 1e-9 else np.nan,
            "J": np.nan, "J_gap_bp": np.nan, "match_oracle": np.nan,
        })

        print(f"  inst={inst}: oracle IS={annualized_sharpe(Rtr, S_star):+.3f} "
              f"OOS={annualized_sharpe(Rte, S_star):+.3f}  "
              f"slh-matches-oracle={int(np.array_equal(np.sort(S_slh), np.sort(S_star)))}  "
              f"eqn OOS={sh_oos_eq:+.3f}  oracle_t={oracle_dt:.1f}s")

results = pd.DataFrame(rows)
results.to_csv(DATA_DIR / "oos_sharpe_results.csv", index=False)
print(f"\nTotal rows: {len(results)}; saved oos_sharpe_results.csv")

## Aggregate and report

In [ ]:
agg = results.groupby(["n", "k", "alg"]).agg(
    sharpe_is_mean=("sharpe_is", "mean"),
    sharpe_oos_mean=("sharpe_oos", "mean"),
    sharpe_oos_se_mean=("sharpe_oos_se", "mean"),
    delta_rel_mean=("delta_rel", "mean"),
    match_oracle_rate=("match_oracle", "mean"),
).reset_index()
agg.to_csv(DATA_DIR / "oos_sharpe_agg.csv", index=False)

alg_order = ["oracle", "slh", "topmu", "eqn"]
agg["alg"] = pd.Categorical(agg["alg"], categories=alg_order, ordered=True)
agg = agg.sort_values(["n", "k", "alg"]).reset_index(drop=True)
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print(agg.to_string(index=False))

## Pooled summary across cells

In [ ]:
print("Pooled means across all cells and instances:\n")
print(f"{'alg':<8}  {'IS':>6}  {'OOS':>6}  {'drop':>6}  {'match':>6}")
for alg in alg_order:
    sub = results[results.alg == alg]
    drop = 100 * sub.delta_rel.mean() if not sub.delta_rel.isna().all() else np.nan
    match = sub.match_oracle.mean() if not sub.match_oracle.isna().all() else np.nan
    print(f"{alg:<8}  {sub.sharpe_is.mean():>+6.3f}  {sub.sharpe_oos.mean():>+6.3f}  "
          f"{drop:>+5.0f}%  {match if not np.isnan(match) else '   -':>6}")

## Figure: two-panel summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

cells = GRID
cell_labels = [f"({n},{k})" for n, k in cells]
alg_labels = {"oracle": "Oracle", "slh": "SLH", "topmu": r"Top-$k$-$\hat\mu$", "eqn": r"Eq-$n$"}
colors = {"oracle": "#1f77b4", "slh": "#d62728", "topmu": "#9467bd", "eqn": "#2ca02c"}

ax = axes[0]
x = np.arange(len(cells))
width = 0.20
for i, alg in enumerate(alg_order):
    means, ses = [], []
    for (n, k) in cells:
        row = agg[(agg.n == n) & (agg.k == k) & (agg.alg == alg)].iloc[0]
        means.append(row.sharpe_oos_mean)
        ses.append(row.sharpe_oos_se_mean)
    ax.bar(x + (i - 1.5) * width, means, width, yerr=ses, label=alg_labels[alg],
           color=colors[alg], edgecolor="black", capsize=2)
ax.set_xticks(x); ax.set_xticklabels(cell_labels)
ax.set_xlabel(r"$(n,k)$"); ax.set_ylabel("Out-of-sample annualized Sharpe")
ax.set_title("OOS Sharpe across cells (Lo SE error bars)")
ax.legend(fontsize=9, loc="upper right")
ax.axhline(0, color="black", lw=0.5); ax.grid(alpha=0.3, axis="y")

ax = axes[1]
for alg in alg_order:
    is_vals, oos_vals = [], []
    for (n, k) in cells:
        row = agg[(agg.n == n) & (agg.k == k) & (agg.alg == alg)].iloc[0]
        is_vals.append(row.sharpe_is_mean); oos_vals.append(row.sharpe_oos_mean)
    ax.plot(is_vals, oos_vals, "o-", label=alg_labels[alg], color=colors[alg], ms=10, lw=1.5)
ax.plot([0.4, 1.3], [0.4, 1.3], "k--", lw=0.8, alpha=0.5, label="IS = OOS")
ax.set_xlabel("In-sample Sharpe"); ax.set_ylabel("Out-of-sample Sharpe")
ax.set_title("IS vs OOS (points below diagonal = overfitting)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "oos_sharpe.png", dpi=130, bbox_inches="tight")
plt.show()

## Headline findings (for the paper)

1. **Oracle relative Sharpe drop:** 30-53% across cells, pooled 43%.
   Confirms the central claim of Section 7.

2. **SLH matches the oracle in OOS Sharpe within one SE on every cell**,
   and exactly equal on 4/5 cells (the (40,5) cell has 1/5 instances where
   SLH's seed picks a non-oracle support, costing 0.07 in OOS Sharpe).

3. **Eq-n beats or ties the oracle on 4/5 cells** (pooled 0.64 vs 0.56).
   It is the only strategy whose OOS Sharpe is not statistically below its
   IS Sharpe (by construction: nothing to overfit). The cardinality-constrained
   optimum on $(\hat\mu, \hat\Sigma)$ does not clear the no-selection baseline.

4. **The covariance penalty buys something but not enough.** Oracle OOS 0.56
   vs Top-k-mu OOS 0.40 is a 0.16 improvement worth ~3 SEs, confirming
   covariance information is real; but neither beats Eq-n.